In [ ]:
!pip install -q -U openai huggingface_hub datasets pandas orjson tqdm jsonlines

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

In [ ]:
# ----------------------------
# Imports
# ----------------------------
import json
import os
import random
import time
import math
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import jsonlines
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ----------------------------
# Secrets
# ----------------------------
OPENAI_API_KEY = userdata.get("openai_api_key")
HF_TOKEN = userdata.get("HF_TOKEN")

if not OPENAI_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY in Colab Secrets.")

client = OpenAI(api_key=OPENAI_API_KEY)

# ----------------------------
# Config
# ----------------------------
BASE_DIR = Path("/content/drive/MyDrive/Speciale/Data generation/Complexity evolved self instruct")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Final desired number of examples in the flat dataset
TARGET_OBSERVATIONS = 20_000

# Number of task families per generation request.
# Each family should yield about 2 final rows:
#   - 1 simple instruction
#   - 1 complex instruction
TASK_BATCH_SIZE = 20

MODEL_TASKS = "gpt-4.1-mini"
MODEL_SOLUTIONS = "gpt-4.1-mini"
TASK_TEMPERATURE = 0.9
SOLUTION_TEMPERATURE = 0.2
RANDOM_SEED = 200

SEED_FILENAME = "Seed file BEST200.jsonl"
SEED_PATH = BASE_DIR / SEED_FILENAME

OUTPUT_DIR = BASE_DIR / "complexity_evolved_openai_batch_out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK_REQUESTS_PATH = OUTPUT_DIR / "task_requests.jsonl"
TASK_OUTPUTS_PATH = OUTPUT_DIR / "task_outputs.jsonl"
TASK_ERRORS_PATH = OUTPUT_DIR / "task_errors.jsonl"
TASKS_PATH = OUTPUT_DIR / "candidate_task_pairs.jsonl"

SOLUTION_REQUESTS_PATH = OUTPUT_DIR / "solution_requests.jsonl"
SOLUTION_OUTPUTS_PATH = OUTPUT_DIR / "solution_outputs.jsonl"
SOLUTION_ERRORS_PATH = OUTPUT_DIR / "solution_errors.jsonl"
DATASET_PATH = OUTPUT_DIR / "complexity_evolved_dataset.jsonl"
CLEAN_DATASET_PATH = OUTPUT_DIR / "complexity_evolved_dataset_clean.jsonl"
SFT_DATASET_PATH = OUTPUT_DIR / "complexity_evolved_sft.jsonl"

TASK_BATCH_META_PATH = OUTPUT_DIR / "task_batch_meta.json"
SOLUTION_BATCH_META_PATH = OUTPUT_DIR / "solution_batch_meta.json"

random.seed(RANDOM_SEED)

In [ ]:
# ----------------------------
# Helper functions
# ----------------------------

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    rows = []
    with jsonlines.open(path, "r") as reader:
        for row in reader:
            rows.append(row)
    return rows


def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(path, "w") as writer:
        writer.write_all(rows)


def save_json(path: Path, obj: Dict[str, Any]) -> None:
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False))


def load_json(path: Path) -> Dict[str, Any]:
    return json.loads(path.read_text())


def sample_seed_text(seed_rows: List[Dict[str, Any]], k: int = 8) -> str:
    k = min(k, len(seed_rows))
    sampled = random.sample(seed_rows, k=k)
    lines = []
    for i, row in enumerate(sampled, start=1):
        task = row.get("instruction") or row.get("task") or row.get("prompt") or str(row)
        lines.append(f"{i}. {task}")
    return "\n".join(lines)


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    candidate = match.group(0)
    try:
        return json.loads(candidate)
    except Exception:
        return None


def normalize_instruction(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip())


def complexity_gain_ok(simple_instruction: str, complex_instruction: str) -> bool:
    """
    Lightweight guard to reject weak 'complex' variants that are just paraphrases.
    """
    s = simple_instruction.strip().lower()
    c = complex_instruction.strip().lower()

    if not s or not c:
        return False

    if s == c:
        return False

    # usually the complex instruction should be more detailed
    if len(c) <= len(s):
        return False

    markers = [
        "edge case", "edge cases", "constraint", "constraints", "optimize",
        "efficient", "efficiency", "validate", "validation", "handle",
        "additional", "multiple", "nested", "sorted", "case-insensitive",
        "deduplicate", "return both", "raise", "error", "robust",
        "time complexity", "space complexity"
    ]
    return any(m in c for m in markers)


def make_task_prompt(seed_text: str, n_pairs: int) -> str:
    return f"""You are generating synthetic Python coding tasks for instruction tuning.

Create {n_pairs} task families.

For each family, produce:
1. one simple Python coding instruction
2. one more complex version of the SAME task

Requirements:
- Both versions must be Python-related.
- The complex version must preserve the same core task, but increase difficulty meaningfully.
- Complexity can be increased by adding constraints, edge cases, stricter validation, efficiency requirements, more realistic input handling, or multi-step reasoning.
- Do NOT make the complex version a completely different task.
- Keep both versions self-contained and solvable in one response.
- Avoid internet, external APIs, GUI interaction, or real file dependencies unless fully simulated.
- Avoid duplicates and near-duplicates across families.
- Mix domains: algorithms, parsing, debugging, data structures, validation, text processing, utilities.

Seed examples:
{seed_text}

Return ONLY valid JSON with this schema:
{{
  "task_pairs": [
    {{
      "family_id": "pair-0001",
      "simple_instruction": "simple task here",
      "complex_instruction": "harder evolved task here"
    }}
  ]
}}
""".strip()


def make_solution_prompt(instruction: str, family_id: str, variant: str) -> str:
    return f"""You are creating a high-quality Python training example.

Write a strong response to the instruction below.

Requirements:
- Prefer correct, runnable Python when code is appropriate.
- Keep the answer self-contained.
- No markdown code fences.
- If explanation is useful, keep it concise.
- Fully satisfy all stated constraints in the instruction.

Instruction:
{instruction}

Return ONLY valid JSON with this schema:
{{
  "family_id": "{family_id}",
  "variant": "{variant}",
  "instruction": "{instruction}",
  "response": "your answer here"
}}
""".strip()


# ----------------------------
# Request builders
# ----------------------------

def build_task_request_line(custom_id: str, seed_text: str, n_pairs: int) -> Dict[str, Any]:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": MODEL_TASKS,
            "temperature": TASK_TEMPERATURE,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You create paired Python coding tasks with a simple version and "
                        "a meaningfully more complex evolved version. Return strict JSON only."
                    ),
                },
                {
                    "role": "user",
                    "content": make_task_prompt(seed_text, n_pairs),
                },
            ],
            "response_format": {"type": "json_object"},
        },
    }


def build_solution_request_line(
    custom_id: str,
    instruction: str,
    family_id: str,
    variant: str
) -> Dict[str, Any]:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": MODEL_SOLUTIONS,
            "temperature": SOLUTION_TEMPERATURE,
            "messages": [
                {
                    "role": "system",
                    "content": "You solve Python instruction tasks and return strict JSON only."
                },
                {
                    "role": "user",
                    "content": make_solution_prompt(instruction, family_id, variant)
                },
            ],
            "response_format": {"type": "json_object"},
        },
    }


# ----------------------------
# Batch construction
# ----------------------------

def build_task_requests(
    seed_path: Path = SEED_PATH,
    output_path: Path = TASK_REQUESTS_PATH,
    target_observations: int = TARGET_OBSERVATIONS,
    task_batch_size: int = TASK_BATCH_SIZE
) -> int:
    seed_rows = load_jsonl(seed_path)
    if not seed_rows:
        raise ValueError("Seed file is empty.")

    # Each task family should become about 2 final examples:
    # simple + complex
    target_families = math.ceil(target_observations / 2)
    n_requests = math.ceil(target_families / task_batch_size)

    rows = []
    for i in range(n_requests):
        seed_text = sample_seed_text(seed_rows, k=min(8, len(seed_rows)))
        rows.append(
            build_task_request_line(
                custom_id=f"taskgen-{i:06d}",
                seed_text=seed_text,
                n_pairs=task_batch_size,
            )
        )

    write_jsonl(output_path, rows)
    return n_requests


def build_solution_requests(
    tasks_path: Path = TASKS_PATH,
    output_path: Path = SOLUTION_REQUESTS_PATH
) -> int:
    task_rows = load_jsonl(tasks_path)
    rows = []

    for i, row in enumerate(task_rows):
        family_id = row["family_id"]

        rows.append(
            build_solution_request_line(
                custom_id=f"solve-simple-{i:06d}",
                instruction=row["simple_instruction"],
                family_id=family_id,
                variant="simple",
            )
        )

        rows.append(
            build_solution_request_line(
                custom_id=f"solve-complex-{i:06d}",
                instruction=row["complex_instruction"],
                family_id=family_id,
                variant="complex",
            )
        )

    write_jsonl(output_path, rows)
    return len(rows)


# ----------------------------
# Batch API utilities
# ----------------------------

def upload_batch_input(jsonl_path: Path):
    with open(jsonl_path, "rb") as f:
        return client.files.create(file=f, purpose="batch")


def create_batch(input_file_id: str, description: str):
    return client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": description},
    )


def submit_batch(jsonl_path: Path, meta_path: Path, description: str) -> Dict[str, Any]:
    input_file = upload_batch_input(jsonl_path)
    batch = create_batch(input_file.id, description=description)
    meta = {
        "batch_id": batch.id,
        "input_file_id": input_file.id,
        "status": batch.status,
        "description": description,
        "submitted_at_unix": int(time.time()),
        "request_file": str(jsonl_path),
    }
    save_json(meta_path, meta)
    return meta


def get_batch(batch_id: str):
    return client.batches.retrieve(batch_id)


def show_batch_status(meta_path: Path) -> Any:
    meta = load_json(meta_path)
    batch = get_batch(meta["batch_id"])
    print("batch_id:", batch.id)
    print("status:", batch.status)
    print("endpoint:", batch.endpoint)
    print("input_file_id:", batch.input_file_id)
    print("output_file_id:", getattr(batch, "output_file_id", None))
    print("error_file_id:", getattr(batch, "error_file_id", None))
    print("request_counts:", getattr(batch, "request_counts", None))
    return batch


def download_file(file_id: str, destination: Path) -> Path:
    content = client.files.content(file_id).content
    destination.write_bytes(content)
    return destination


def download_batch_artifacts(
    meta_path: Path,
    output_path: Path,
    error_path: Path
) -> Dict[str, Optional[str]]:
    meta = load_json(meta_path)
    batch = get_batch(meta["batch_id"])

    output_file_id = getattr(batch, "output_file_id", None)
    error_file_id = getattr(batch, "error_file_id", None)

    if output_file_id:
        download_file(output_file_id, output_path)
        print("Downloaded output file to:", output_path)
    else:
        print("No output file available yet.")

    if error_file_id:
        download_file(error_file_id, error_path)
        print("Downloaded error file to:", error_path)
    else:
        print("No error file available.")

    return {
        "status": batch.status,
        "output_file_id": output_file_id,
        "error_file_id": error_file_id,
    }


def parse_chat_batch_line(line: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    try:
        body = line["response"]["body"]
        content = body["choices"][0]["message"]["content"]
        payload = extract_json_object(content)
        return payload
    except Exception:
        return None


# ----------------------------
# Parsing + flattening
# ----------------------------

def parse_task_outputs(
    output_path: Path = TASK_OUTPUTS_PATH,
    tasks_path: Path = TASKS_PATH
) -> pd.DataFrame:
    if not output_path.exists():
        raise FileNotFoundError(f"Task output file not found: {output_path}")

    accepted = []
    seen_simple = set()
    seen_complex = set()
    family_counter = 0

    with jsonlines.open(output_path, "r") as reader:
        for row in reader:
            payload = parse_chat_batch_line(row)
            if not payload:
                continue

            pairs = payload.get("task_pairs", [])
            if not isinstance(pairs, list):
                continue

            for pair in pairs:
                if not isinstance(pair, dict):
                    continue

                simple_instruction = normalize_instruction(str(pair.get("simple_instruction", "")))
                complex_instruction = normalize_instruction(str(pair.get("complex_instruction", "")))

                if len(simple_instruction) < 20 or len(complex_instruction) < 20:
                    continue

                if not complexity_gain_ok(simple_instruction, complex_instruction):
                    continue

                simple_key = simple_instruction.lower()
                complex_key = complex_instruction.lower()

                if simple_key in seen_simple or complex_key in seen_complex:
                    continue

                if simple_key == complex_key:
                    continue

                family_counter += 1
                family_id = str(pair.get("family_id", "")).strip() or f"pair-{family_counter:06d}"

                seen_simple.add(simple_key)
                seen_complex.add(complex_key)

                accepted.append({
                    "family_id": family_id,
                    "simple_instruction": simple_instruction,
                    "complex_instruction": complex_instruction,
                })

    write_jsonl(tasks_path, accepted)
    df = pd.DataFrame(accepted)
    print("Accepted task families:", len(df))
    print("Expected final flat rows after solving:", len(df) * 2)
    return df


def parse_solution_outputs(
    output_path: Path = SOLUTION_OUTPUTS_PATH,
    dataset_path: Path = DATASET_PATH
) -> pd.DataFrame:
    """
    Important:
    This FLATTENS the simple/complex families into plain rows.
    The written dataset only contains:
      - instruction
      - response
    """
    if not output_path.exists():
        raise FileNotFoundError(f"Solution output file not found: {output_path}")

    flat_rows = []

    with jsonlines.open(output_path, "r") as reader:
        for row in reader:
            payload = parse_chat_batch_line(row)
            if not payload:
                continue

            instruction = str(payload.get("instruction", "")).strip()
            response = str(payload.get("response", "")).strip()

            if not instruction or not response:
                continue

            flat_rows.append({
                "instruction": normalize_instruction(instruction),
                "response": response,
            })

    write_jsonl(dataset_path, flat_rows)
    df = pd.DataFrame(flat_rows)
    print("Flat dataset rows:", len(df))
    return df


def clean_dataset(
    dataset_path: Path = DATASET_PATH,
    clean_path: Path = CLEAN_DATASET_PATH
) -> pd.DataFrame:
    """
    Final clean output remains FLAT:
      {"instruction": ..., "response": ...}
    """
    rows = load_jsonl(dataset_path)
    cleaned = []
    seen = set()

    for row in rows:
        instruction = normalize_instruction(row["instruction"])
        response = row["response"].strip()

        if len(instruction) < 20 or len(response) < 10:
            continue

        key = instruction.lower()
        if key in seen:
            continue
        seen.add(key)

        cleaned.append({
            "instruction": instruction,
            "response": response,
        })

    write_jsonl(clean_path, cleaned)
    df = pd.DataFrame(cleaned)
    print("Clean flat rows:", len(df))
    return df


def convert_to_sft(
    clean_path: Path = CLEAN_DATASET_PATH,
    sft_path: Path = SFT_DATASET_PATH
) -> pd.DataFrame:
    rows = load_jsonl(clean_path)
    sft_rows = []

    for row in rows:
        sft_rows.append({
            "messages": [
                {"role": "user", "content": row["instruction"]},
                {"role": "assistant", "content": row["response"]},
            ]
        })

    write_jsonl(sft_path, sft_rows)
    df = pd.DataFrame({"messages": [r["messages"] for r in sft_rows]})
    print("SFT rows:", len(df))
    return df


# ----------------------------
# Hugging Face upload
# ----------------------------

def upload_dataset_to_hf(
    repo_id: str,
    source_path: Path = CLEAN_DATASET_PATH,
    private: bool = False,
    commit_message: str = "Upload complexity-evolved self-instruct dataset"
) -> None:
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN not found in Colab Secrets.")

    from huggingface_hub import HfApi

    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=repo_id, repo_type="dataset", private=private, exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(source_path),
        path_in_repo=source_path.name,
        repo_id=repo_id,
        repo_type="dataset",
        commit_message=commit_message,
    )
    print(f"Uploaded {source_path.name} to hf.co/datasets/{repo_id}")

## Phase 1: Build and submit the task-generation batch

In [ ]:
n_task_requests = build_task_requests()
print("Task-generation requests written:", n_task_requests)
print("Path:", TASK_REQUESTS_PATH)

In [ ]:
task_meta = submit_batch(
    jsonl_path=TASK_REQUESTS_PATH,
    meta_path=TASK_BATCH_META_PATH,
    description="complexity-evolved self-instruct task generation",
)
task_meta

In [ ]:
show_batch_status(TASK_BATCH_META_PATH)

In [ ]:
download_batch_artifacts(
    meta_path=TASK_BATCH_META_PATH,
    output_path=TASK_OUTPUTS_PATH,
    error_path=TASK_ERRORS_PATH,
)

task_df = parse_task_outputs(
    output_path=TASK_OUTPUTS_PATH,
    tasks_path=TASKS_PATH,
)

task_df.head()

In [ ]:
print("Accepted task families:", len(load_jsonl(TASKS_PATH)))
print("Target final observations:", TARGET_OBSERVATIONS)
print("Expected final rows before cleaning:", len(load_jsonl(TASKS_PATH)) * 2)
print("Coverage ratio (approx):", round((len(load_jsonl(TASKS_PATH)) * 2) / TARGET_OBSERVATIONS, 3))

## Phase 2: Build and submit the solution-generation batch

In [ ]:
n_solution_requests = build_solution_requests()
print("Solution-generation requests written:", n_solution_requests)
print("Path:", SOLUTION_REQUESTS_PATH)

In [ ]:
solution_meta = submit_batch(
    jsonl_path=SOLUTION_REQUESTS_PATH,
    meta_path=SOLUTION_BATCH_META_PATH,
    description="complexity-evolved self-instruct solution generation",
)
solution_meta

In [ ]:
show_batch_status(SOLUTION_BATCH_META_PATH)

In [ ]:
download_batch_artifacts(
    meta_path=SOLUTION_BATCH_META_PATH,
    output_path=SOLUTION_OUTPUTS_PATH,
    error_path=SOLUTION_ERRORS_PATH,
)

dataset_df = parse_solution_outputs(
    output_path=SOLUTION_OUTPUTS_PATH,
    dataset_path=DATASET_PATH,
)

dataset_df.head()

## Phase 3: Clean and convert to SFT format

In [ ]:
clean_df = clean_dataset()
sft_df = convert_to_sft()

print("Raw flat dataset rows:", len(load_jsonl(DATASET_PATH)))
print("Clean flat dataset rows:", len(load_jsonl(CLEAN_DATASET_PATH)))
print("SFT dataset rows:", len(load_jsonl(SFT_DATASET_PATH)))

In [ ]:
pd.DataFrame(load_jsonl(CLEAN_DATASET_PATH)[:10])

## Optional: Upload the final flat dataset to Hugging Face

In [ ]:
# upload_dataset_to_hf(
#     repo_id="your-username/complexity-evolved-self-instruct-python-20k",
#     source_path=CLEAN_DATASET_PATH,
#     private=False,
# )

print("Uncomment and edit the function call above when you are ready.")